# Robustness Metrics Analysis

This notebook analyzes robustness using:
- **Degradation Rate**: Relative drop in performance (clamped at 0)
- **Consistency**: Penalizes any change (both improvement and degradation)

**IMPORTANT: CLIP excluded from robustness evaluation**

CLIP score is excluded because it shows opposite-direction behavior (perturbed > clean for most models).
This happens because perturbation causes simpler HTML generation, which CLIP rates as "more similar" to the reference.

**Robustness Score = (IoU + Structural Alignment) / 2**
- IoU: Layout preservation
- Structural Alignment = (Semantic HTML + Tree Edit) / 2

In [1]:
import json
import numpy as np
from scipy.stats import spearmanr, pearsonr
from pathlib import Path

BASE_DIR = Path('/root/Design2code/robustness_results')

In [2]:
def load_robustness_data(model_dir):
    """Load robustness data from JSON file."""
    for filename in ['robustness_report.json', 'robustness_results.json']:
        json_path = model_dir / filename
        if json_path.exists():
            with open(json_path) as f:
                return json.load(f)
    return None

def print_current_report(model_dir):
    """Print the current robustness report text file."""
    txt_path = model_dir / 'robustness_report.txt'
    if txt_path.exists():
        print("=" * 70)
        print(f"CURRENT REPORT: {model_dir.name}")
        print("=" * 70)
        with open(txt_path) as f:
            print(f.read())
    else:
        print(f"No robustness_report.txt found in {model_dir}")

In [3]:
def compute_per_sample_combined(per_sample):
    """
    Compute per-sample combined scores.
    
    NOTE: CLIP is EXCLUDED from robustness evaluation because it shows
    opposite-direction behavior (perturbed > clean for most models).
    
    Robustness Score = (IoU + Structural Alignment) / 2
    """
    n = len(per_sample.get('clean_iou', per_sample.get('clean_clip', [])))
    
    # IoU only (CLIP excluded)
    clean_iou = np.array(per_sample.get('clean_iou', [0]*n))
    perturbed_iou = np.array(per_sample.get('perturbed_iou', [0]*n))
    
    # Structural Alignment
    clean_sa = np.array(per_sample.get('clean_structural', [0]*n))
    perturbed_sa = np.array(per_sample.get('perturbed_structural', [0]*n))
    
    # Robustness Overall = (IoU + Structural) / 2
    clean_overall = (clean_iou + clean_sa) / 2
    perturbed_overall = (perturbed_iou + perturbed_sa) / 2
    
    # Also get CLIP for reference (but not used in robustness score)
    clean_clip = np.array(per_sample.get('clean_clip', [0]*n))
    perturbed_clip = np.array(per_sample.get('perturbed_clip', [0]*n))
    
    return {
        'clean_iou': clean_iou,
        'perturbed_iou': perturbed_iou,
        'clean_sa': clean_sa,
        'perturbed_sa': perturbed_sa,
        'clean_overall': clean_overall,
        'perturbed_overall': perturbed_overall,
        'clean_clip': clean_clip,
        'perturbed_clip': perturbed_clip,
    }

In [4]:
def compute_consistency(clean, perturbed):
    """
    Consistency: 1 - |clean - perturbed| / max(clean, perturbed)
    Penalizes ANY change (both improvement and degradation).
    Range: [0, 1], higher is better.
    """
    clean = np.array(clean)
    perturbed = np.array(perturbed)
    
    abs_diff = np.abs(clean - perturbed)
    max_scores = np.maximum(clean, perturbed)
    
    per_sample = np.where(max_scores > 0, 1 - abs_diff / max_scores, 1.0)
    
    return {
        'mean': float(np.mean(per_sample)),
        'std': float(np.std(per_sample)),
        'min': float(np.min(per_sample))
    }

def compute_degradation_rate(clean, perturbed):
    """
    Degradation Rate: max(0, (clean - perturbed) / clean)
    Only penalizes drops, improvements clamped to 0.
    Range: [0, 1], lower is better.
    """
    clean = np.array(clean)
    perturbed = np.array(perturbed)
    
    clean_mean = np.mean(clean)
    perturbed_mean = np.mean(perturbed)
    
    if clean_mean > 0:
        degradation = max(0, (clean_mean - perturbed_mean) / clean_mean)
    else:
        degradation = 0.0
    
    return float(degradation)

def compute_correlation(clean, perturbed):
    """Spearman correlation (rank preservation)."""
    if len(clean) < 3 or np.std(clean) == 0 or np.std(perturbed) == 0:
        return None
    corr, _ = spearmanr(clean, perturbed)
    return float(corr)

In [5]:
def get_rating(score):
    if score >= 0.95: return "EXCELLENT"
    elif score >= 0.85: return "GOOD"
    elif score >= 0.70: return "MODERATE"
    elif score >= 0.50: return "POOR"
    else: return "UNRELIABLE"

In [6]:
def analyze_model(model_name, model_dir):
    """
    Analyze model robustness.
    
    Robustness Score = (IoU + Structural) / 2
    CLIP is excluded due to opposite-direction behavior.
    """
    data = load_robustness_data(model_dir)
    if data is None:
        print(f"No data found for {model_name}")
        return None
    
    if 'per_sample' not in data:
        print(f"{model_name}: No per-sample data available")
        return None
    
    combined = compute_per_sample_combined(data['per_sample'])
    n = len(combined['clean_overall'])
    
    return {
        'model': model_name,
        'samples': n,
        'strength': data.get('strength', 0.2),
        
        # Robustness Overall = (IoU + Structural) / 2  [CLIP excluded]
        'clean_mean': float(np.mean(combined['clean_overall'])),
        'perturbed_mean': float(np.mean(combined['perturbed_overall'])),
        'consistency': compute_consistency(combined['clean_overall'], combined['perturbed_overall']),
        'degradation_rate': compute_degradation_rate(combined['clean_overall'], combined['perturbed_overall']),
        'correlation': compute_correlation(combined['clean_overall'], combined['perturbed_overall']),
        
        # Breakdown by component
        'iou_clean': float(np.mean(combined['clean_iou'])),
        'iou_perturbed': float(np.mean(combined['perturbed_iou'])),
        'iou_consistency': compute_consistency(combined['clean_iou'], combined['perturbed_iou'])['mean'],
        'sa_clean': float(np.mean(combined['clean_sa'])),
        'sa_perturbed': float(np.mean(combined['perturbed_sa'])),
        'sa_consistency': compute_consistency(combined['clean_sa'], combined['perturbed_sa'])['mean'],
        
        # CLIP for reference (excluded from robustness)
        'clip_clean': float(np.mean(combined['clean_clip'])),
        'clip_perturbed': float(np.mean(combined['perturbed_clip'])),
        'clip_change': float(np.mean(combined['perturbed_clip']) - np.mean(combined['clean_clip'])),
    }

In [7]:
def print_model_analysis(r):
    """Print analysis for one model."""
    if r is None:
        return
    
    print("\n" + "=" * 70)
    print(f"ROBUSTNESS ANALYSIS: {r['model'].upper()}")
    print(f"Samples: {r['samples']}, Strength: {r['strength']}")
    print("=" * 70)
    
    # Overall scores
    change = r['perturbed_mean'] - r['clean_mean']
    rating = get_rating(r['consistency']['mean'])
    
    print(f"\nROBUSTNESS SCORE = (IoU + Structural) / 2  [CLIP excluded]")
    print(f"  Clean:     {r['clean_mean']:.4f}")
    print(f"  Perturbed: {r['perturbed_mean']:.4f}  ({change:+.4f})")
    
    print(f"\n  DEGRADATION RATE: {r['degradation_rate']*100:.2f}%")
    print(f"  CONSISTENCY:      {r['consistency']['mean']:.4f} ({rating})")
    print(f"    Std: {r['consistency']['std']:.4f}, Min: {r['consistency']['min']:.4f}")
    
    if r['correlation']:
        print(f"\n  CORRELATION (Spearman): {r['correlation']:.4f}")
    
    # Breakdown
    print(f"\n  BREAKDOWN:")
    print(f"    IoU:        {r['iou_clean']:.4f} -> {r['iou_perturbed']:.4f}  (consistency: {r['iou_consistency']:.4f})")
    print(f"    Structural: {r['sa_clean']:.4f} -> {r['sa_perturbed']:.4f}  (consistency: {r['sa_consistency']:.4f})")
    
    # CLIP reference
    print(f"\n  CLIP (excluded from robustness - opposite direction):")
    print(f"    {r['clip_clean']:.4f} -> {r['clip_perturbed']:.4f}  ({r['clip_change']:+.4f})")

---
## Qwen 0.2

In [8]:
print_current_report(BASE_DIR / 'qwen_0.2')

CURRENT REPORT: qwen_0.2
QWEN MODEL - ROBUSTNESS EVALUATION REPORT
Generated: 2025-12-17 05:28:33

CONFIGURATION:
  Model:              qwen
  Total Samples:      50
  Perturbation:       Crop + Brightness + Contrast + Blur
  Perturbation Strength: 0.2

1. VISUAL FIDELITY (CLIP + IoU)

Combined Visual Fidelity = (CLIP + IoU) / 2
  Clean:     0.4056
  Perturbed: 0.4453

1.1 CLIP Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.6734
    Std:    0.1477
    Count:  50
  Perturbed Predictions:
    Mean:   0.7619
    Std:    0.1488
    Count:  50

1.2 IoU Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.1379
    Std:    0.0778
    Count:  50
  Perturbed Predictions:
    Mean:   0.1287
    Std:    0.0619
    Count:  50

2. STRUCTURAL ALIGNMENT (Semantic + Accessibility + Tree Edit)

Combined Score = (Semantic + Accessibility + Tree Edit) / 3

  Clean Predictions:
    Combined Mean:   0.1956
    Std:             0.1329
    Count:           

In [9]:
qwen = analyze_model('qwen_0.2', BASE_DIR / 'qwen_0.2')
print_model_analysis(qwen)


ROBUSTNESS ANALYSIS: QWEN_0.2
Samples: 50, Strength: 0.2

ROBUSTNESS SCORE = (IoU + Structural) / 2  [CLIP excluded]
  Clean:     0.1667
  Perturbed: 0.1316  (-0.0351)

  DEGRADATION RATE: 21.05%
  CONSISTENCY:      0.6575 (POOR)
    Std: 0.2745, Min: 0.0000

  CORRELATION (Spearman): 0.4322

  BREAKDOWN:
    IoU:        0.1379 -> 0.1287  (consistency: 0.6975)
    Structural: 0.1956 -> 0.1346  (consistency: 0.5468)

  CLIP (excluded from robustness - opposite direction):
    0.6734 -> 0.7619  (+0.0885)


/tmp/ipykernel_2516697/4052065280.py:13: RuntimeWarning: invalid value encountered in divide
  per_sample = np.where(max_scores > 0, 1 - abs_diff / max_scores, 1.0)


---
## WebSight 0.2

In [10]:
print_current_report(BASE_DIR / 'websight_0.2')

CURRENT REPORT: websight_0.2
WEBSIGHT MODEL - ROBUSTNESS EVALUATION REPORT
Generated: 2025-12-17 11:08:13

CONFIGURATION:
  Model:              websight
  Total Samples:      50
  Perturbation:       Crop + Brightness + Contrast + Blur
  Perturbation Strength: 0.2

1. VISUAL FIDELITY (CLIP + IoU)

Combined Visual Fidelity = (CLIP + IoU) / 2
  Clean:     0.4033
  Perturbed: 0.3824

1.1 CLIP Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.7034
    Std:    0.1301
    Count:  50
  Perturbed Predictions:
    Mean:   0.6736
    Std:    0.1365
    Count:  50

1.2 IoU Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.1032
    Std:    0.0786
    Count:  50
  Perturbed Predictions:
    Mean:   0.0913
    Std:    0.0669
    Count:  50

2. STRUCTURAL ALIGNMENT (Semantic + Accessibility + Tree Edit)

Combined Score = (Semantic + Accessibility + Tree Edit) / 3

  Clean Predictions:
    Combined Mean:   0.1072
    Std:             0.1146
    Count

In [11]:
websight = analyze_model('websight_0.2', BASE_DIR / 'websight_0.2')
print_model_analysis(websight)


ROBUSTNESS ANALYSIS: WEBSIGHT_0.2
Samples: 50, Strength: 0.2

ROBUSTNESS SCORE = (IoU + Structural) / 2  [CLIP excluded]
  Clean:     0.1052
  Perturbed: 0.0815  (-0.0237)

  DEGRADATION RATE: 22.50%
  CONSISTENCY:      0.5196 (POOR)
    Std: 0.2284, Min: 0.0731

  CORRELATION (Spearman): 0.3234

  BREAKDOWN:
    IoU:        0.1032 -> 0.0913  (consistency: 0.6135)
    Structural: 0.1072 -> 0.0717  (consistency: 0.3907)

  CLIP (excluded from robustness - opposite direction):
    0.7034 -> 0.6736  (-0.0298)


---
## Gemini 0.2

In [12]:
print_current_report(BASE_DIR / 'gemini_0.2')

CURRENT REPORT: gemini_0.2
GEMINI MODEL - ROBUSTNESS EVALUATION REPORT
Generated: 2025-12-18 01:39:40

CONFIGURATION:
  Model:              gemini
  Total Samples:      46
  Perturbation:       Crop + Brightness + Contrast + Blur
  Perturbation Strength: 0.2

1. VISUAL FIDELITY (CLIP + IoU)

Combined Visual Fidelity = (CLIP + IoU) / 2
  Clean:     0.5077
  Perturbed: 0.5189

1.1 CLIP Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.8430
    Std:    0.0893
    Count:  46
  Perturbed Predictions:
    Mean:   0.8552
    Std:    0.0799
    Count:  46

1.2 IoU Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.1724
    Std:    0.1024
    Count:  46
  Perturbed Predictions:
    Mean:   0.1826
    Std:    0.0809
    Count:  46

2. STRUCTURAL ALIGNMENT (Semantic + Tree Edit)

Combined Score = (Semantic + Tree Edit) / 2

  Clean Predictions:
    Combined Mean:   0.2323
    Std:             0.1882
    Count:           46
    - Semantic HTML: 0.

In [13]:
gemini = analyze_model('gemini_0.2', BASE_DIR / 'gemini_0.2')
print_model_analysis(gemini)


ROBUSTNESS ANALYSIS: GEMINI_0.2
Samples: 46, Strength: 0.2

ROBUSTNESS SCORE = (IoU + Structural) / 2  [CLIP excluded]
  Clean:     0.2023
  Perturbed: 0.2187  (+0.0163)

  DEGRADATION RATE: 0.00%
  CONSISTENCY:      0.6620 (POOR)
    Std: 0.3711, Min: 0.0000

  CORRELATION (Spearman): 0.6934

  BREAKDOWN:
    IoU:        0.1724 -> 0.1826  (consistency: 0.6307)
    Structural: 0.2323 -> 0.2548  (consistency: 0.6327)

  CLIP (excluded from robustness - opposite direction):
    0.8430 -> 0.8552  (+0.0122)


---
## Design2Code-18B 0.2

In [14]:
print_current_report(BASE_DIR / 'design2code18b_0.2')

CURRENT REPORT: design2code18b_0.2
DESIGN2CODE-18B MODEL - ROBUSTNESS EVALUATION REPORT
Generated: 2025-12-18 02:29:32

CONFIGURATION:
  Model:              design2code-18b
  Total Samples:      50
  Perturbation:       Crop + Brightness + Contrast + Blur
  Perturbation Strength: 0.2

1. VISUAL FIDELITY (CLIP + IoU)

Combined Visual Fidelity = (CLIP + IoU) / 2
  Clean:     0.4295
  Perturbed: 0.4147

1.1 CLIP Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.7404
    Std:    0.1137
    Count:  50
  Perturbed Predictions:
    Mean:   0.7293
    Std:    0.1098
    Count:  50

1.2 IoU Score (range: [0, 1], higher is better):
  Clean Predictions:
    Mean:   0.1187
    Std:    0.0695
    Count:  50
  Perturbed Predictions:
    Mean:   0.1002
    Std:    0.0536
    Count:  50

2. STRUCTURAL ALIGNMENT (Semantic + Tree Edit)

Combined Score = (Semantic + Tree Edit) / 2

  Clean Predictions:
    (Not computed)

  Perturbed Predictions:
    (Not computed)

3. ROBUSTNES

In [15]:
design2code = analyze_model('design2code18b_0.2', BASE_DIR / 'design2code18b_0.2')
print_model_analysis(design2code)


ROBUSTNESS ANALYSIS: DESIGN2CODE18B_0.2
Samples: 50, Strength: 0.2

ROBUSTNESS SCORE = (IoU + Structural) / 2  [CLIP excluded]
  Clean:     0.0593
  Perturbed: 0.0501  (-0.0092)

  DEGRADATION RATE: 15.56%
  CONSISTENCY:      0.7175 (MODERATE)
    Std: 0.1885, Min: 0.3344

  CORRELATION (Spearman): 0.6896

  BREAKDOWN:
    IoU:        0.1187 -> 0.1002  (consistency: 0.7175)
    Structural: 0.0000 -> 0.0000  (consistency: 1.0000)

  CLIP (excluded from robustness - opposite direction):
    0.7404 -> 0.7293  (-0.0111)


/tmp/ipykernel_2516697/4052065280.py:13: RuntimeWarning: invalid value encountered in divide
  per_sample = np.where(max_scores > 0, 1 - abs_diff / max_scores, 1.0)


---
## Summary Table

In [16]:
def print_summary(models):
    """Summary table with degradation rate and consistency per model."""
    valid = [m for m in models if m is not None]
    
    print("\n" + "=" * 100)
    print("ROBUSTNESS SUMMARY: (IoU + Structural) / 2  [CLIP excluded]")
    print("=" * 100)
    print(f"{'Model':<18} {'Clean':>8} {'Pert':>8} {'Change':>8} {'Degrad%':>10} {'Consistency':>12} {'Rating':<10}")
    print("-" * 100)
    
    for m in valid:
        change = m['perturbed_mean'] - m['clean_mean']
        cons = m['consistency']['mean']
        rating = get_rating(cons)
        deg = m['degradation_rate'] * 100
        
        print(f"{m['model']:<18} {m['clean_mean']:>8.4f} {m['perturbed_mean']:>8.4f} {change:>+8.4f} "
              f"{deg:>9.2f}% {cons:>12.4f} {rating:<10}")
    
    print("\n" + "=" * 100)
    print("INTERPRETATION:")
    print("  Degradation%: Performance drop (0% = no drop, clamped if improved)")
    print("  Consistency:  1.0 = identical outputs | <0.5 = highly unstable")
    print("  Rating: >=0.95 EXCELLENT | >=0.85 GOOD | >=0.70 MODERATE | >=0.50 POOR | <0.50 UNRELIABLE")
    print("=" * 100)

In [17]:
all_models = [qwen, websight, gemini, design2code]
print_summary(all_models)


ROBUSTNESS SUMMARY: (IoU + Structural) / 2  [CLIP excluded]
Model                 Clean     Pert   Change    Degrad%  Consistency Rating    
----------------------------------------------------------------------------------------------------
qwen_0.2             0.1667   0.1316  -0.0351     21.05%       0.6575 POOR      
websight_0.2         0.1052   0.0815  -0.0237     22.50%       0.5196 POOR      
gemini_0.2           0.2023   0.2187  +0.0163      0.00%       0.6620 POOR      
design2code18b_0.2   0.0593   0.0501  -0.0092     15.56%       0.7175 MODERATE  

INTERPRETATION:
  Degradation%: Performance drop (0% = no drop, clamped if improved)
  Consistency:  1.0 = identical outputs | <0.5 = highly unstable
  Rating: >=0.95 EXCELLENT | >=0.85 GOOD | >=0.70 MODERATE | >=0.50 POOR | <0.50 UNRELIABLE


---
## Why CLIP is Excluded

In [18]:
def explain_clip_exclusion(models):
    """Show why CLIP is excluded from robustness evaluation."""
    valid = [m for m in models if m is not None]
    
    print("\n" + "=" * 90)
    print("WHY CLIP IS EXCLUDED FROM ROBUSTNESS")
    print("=" * 90)
    print("\nCLIP shows OPPOSITE direction from other metrics:")
    print("- Perturbation causes simpler HTML generation")
    print("- CLIP rates simpler outputs as 'more similar' to reference")
    print("- This gives false impression of 'improvement' under perturbation")
    print("\n" + "-" * 90)
    print(f"{'Model':<18} {'CLIP':>20} {'IoU':>20} {'Structural':>20}")
    print(f"{'':18} {'Clean->Pert':>20} {'Clean->Pert':>20} {'Clean->Pert':>20}")
    print("-" * 90)
    
    for m in valid:
        clip_dir = "↑ IMPROVED" if m['clip_change'] > 0.01 else ("↓ degraded" if m['clip_change'] < -0.01 else "~ stable")
        iou_change = m['iou_perturbed'] - m['iou_clean']
        iou_dir = "↑ improved" if iou_change > 0.01 else ("↓ DEGRADED" if iou_change < -0.01 else "~ stable")
        sa_change = m['sa_perturbed'] - m['sa_clean']
        sa_dir = "↑ improved" if sa_change > 0.01 else ("↓ DEGRADED" if sa_change < -0.01 else "~ stable")
        
        print(f"{m['model']:<18} {m['clip_change']:>+8.4f} {clip_dir:<10} {iou_change:>+8.4f} {iou_dir:<10} {sa_change:>+8.4f} {sa_dir:<10}")
    
    print("\n" + "=" * 90)
    print("CONCLUSION: CLIP goes opposite direction -> excluded from robustness score")
    print("=" * 90)

explain_clip_exclusion(all_models)


WHY CLIP IS EXCLUDED FROM ROBUSTNESS

CLIP shows OPPOSITE direction from other metrics:
- Perturbation causes simpler HTML generation
- CLIP rates simpler outputs as 'more similar' to reference
- This gives false impression of 'improvement' under perturbation

------------------------------------------------------------------------------------------
Model                              CLIP                  IoU           Structural
                            Clean->Pert          Clean->Pert          Clean->Pert
------------------------------------------------------------------------------------------
qwen_0.2            +0.0885 ↑ IMPROVED  -0.0092 ~ stable    -0.0610 ↓ DEGRADED
websight_0.2        -0.0298 ↓ degraded  -0.0119 ↓ DEGRADED  -0.0354 ↓ DEGRADED
gemini_0.2          +0.0122 ↑ IMPROVED  +0.0101 ↑ improved  +0.0225 ↑ improved
design2code18b_0.2  -0.0111 ↓ degraded  -0.0185 ↓ DEGRADED  +0.0000 ~ stable  

CONCLUSION: CLIP goes opposite direction -> excluded from robustness score


---
## Consistency vs Degradation Rate

In [19]:
def explain_consistency_vs_degradation(models):
    """Explain why consistency catches what degradation rate misses."""
    valid = [m for m in models if m is not None]
    
    print("\n" + "=" * 90)
    print("CONSISTENCY vs DEGRADATION RATE")
    print("=" * 90)
    print("\nCurrent formula: Degradation = max(0, (clean - perturbed) / clean)")
    print("Problem: If perturbed > clean, degradation = 0% (clamped) - looks 'robust'")
    print("\nConsistency formula: 1 - |clean - perturbed| / max(clean, perturbed)")
    print("Better: Penalizes ANY change - both degradation AND anomalous improvement")
    print("\n" + "-" * 90)
    print(f"{'Model':<18} {'Change':>10} {'Direction':<15} {'Degradation%':<15} {'Consistency':<15}")
    print("-" * 90)
    
    for m in valid:
        change = m['perturbed_mean'] - m['clean_mean']
        
        if change > 0.01:
            direction = "Improved"
            deg_pct = "0% (clamped)"
        elif change < -0.01:
            direction = "Degraded"
            deg_pct = f"{abs(change) / m['clean_mean'] * 100:.1f}%"
        else:
            direction = "Stable"
            deg_pct = "~0%"
        
        cons = m['consistency']['mean']
        cons_rating = get_rating(cons)
        
        print(f"{m['model']:<18} {change:>+10.4f} {direction:<15} {deg_pct:<15} {cons:.4f} ({cons_rating})")
    
    print("\n" + "=" * 90)

explain_consistency_vs_degradation(all_models)


CONSISTENCY vs DEGRADATION RATE

Current formula: Degradation = max(0, (clean - perturbed) / clean)
Problem: If perturbed > clean, degradation = 0% (clamped) - looks 'robust'

Consistency formula: 1 - |clean - perturbed| / max(clean, perturbed)
Better: Penalizes ANY change - both degradation AND anomalous improvement

------------------------------------------------------------------------------------------
Model                  Change Direction       Degradation%    Consistency    
------------------------------------------------------------------------------------------
qwen_0.2              -0.0351 Degraded        21.1%           0.6575 (POOR)
websight_0.2          -0.0237 Degraded        22.5%           0.5196 (POOR)
gemini_0.2            +0.0163 Improved        0% (clamped)    0.6620 (POOR)
design2code18b_0.2    -0.0092 Stable          ~0%             0.7175 (MODERATE)

